# Torch Available

In [1]:
import torch

print(torch.cuda.is_available())  # Should return True if CUDA is detected
print(torch.__version__)  # Example output: '2.1.0+cpu' or '2.1.0+cu118'


True
2.6.0+cu124


In [2]:
import torch
print(torch.version.cuda)


12.4


# Test Cartpole Overfit

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
import gym
import procgen

In [5]:
from shimmy.openai_gym_compatibility import GymV26CompatibilityV0
cart_env = GymV26CompatibilityV0("CartPole-v1", make_kwargs={"render_mode": "rgb_array"})

In [6]:
cart_env.reset()

(array([ 0.02983963, -0.01658122, -0.03890304,  0.00212702], dtype=float32),
 {})

In [7]:
from stable_baselines3.common import env_checker
env_checker.check_env(cart_env)

/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


In [9]:
observation, info = cart_env.reset(seed=42)
total_reward = 0
for _ in range(1000):
    action = cart_env.action_space.sample()  # this is where you would insert your policy
    observation, reward, terminated, truncated, info = cart_env.step(action)
    total_reward += reward
    if terminated or truncated:
        observation, info = cart_env.reset()
cart_env.close()
print(total_reward)

1000.0


In [8]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model = PPO("MlpPolicy", cart_env, verbose=1, device="cuda")
model.learn(5_000_000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


KeyboardInterrupt: 

# PRocgen

In [6]:
from shimmy.openai_gym_compatibility import GymV21CompatibilityV0
# cart_env = GymV26CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={"render_mode": "rgb_array"})
coin_non_env = GymV21CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={"num_levels": 1, "start_level": 0, "distribution_mode": "easy"})


In [7]:
import gymnasium as gym
from procgen import ProcgenEnv
from shimmy.openai_gym_compatibility import GymV21CompatibilityV0
from stable_baselines3.common import env_checker

# Custom wrapper to ensure compatibility with Stable Baselines3 and the 'seed' parameter
class CustomProcgenEnvWrapper(gym.Env):
    def __init__(self, env):
        self.env = env

    def reset(self, *args, **kwargs):
        # Ignore the seed parameter and just call the original reset method
        if 'seed' in kwargs:
            del kwargs['seed']  # Ignore the seed parameter, but still accept it
        return self.env.reset(*args, **kwargs)

    def step(self, action):
        state, reward, terminated, truncated, info =  self.env.step(action)
        reward = float(reward)
        terminated = bool(terminated)
        truncated = bool(truncated)
        return (state, reward, terminated , truncated, info)

    def render(self):
        return self.env.render()

    @property
    def observation_space(self):
        return self.env.observation_space
    
    @property
    def action_space(self):
        return self.env.action_space

# Create the original Procgen environment
coin_non_env = GymV21CompatibilityV0("procgen:procgen-coinrun-v0", make_kwargs={
    "num_levels": 10,
    "start_level": 0,
    "distribution_mode": "easy"
})

# Apply the custom wrapper
wrapped_env = CustomProcgenEnvWrapper(coin_non_env)

# # Optionally, vectorize the environment
# from stable_baselines3.common.vec_env import DummyVecEnv
# vec_env = DummyVecEnv([lambda: wrapped_env])

env_checker.check_env(wrapped_env)
# Now you can train your model, with `reset()` accepting `seed` but ignoring it


/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'numpy.ndarray'>`
  logger.warn(
/storage/home/hcoda1/

In [12]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model = PPO("MlpPolicy", wrapped_env, verbose=1, device="cuda")
model.learn(5_000_000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/rl_env2/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 413      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    fps             | 446      |
|    iterations      | 1        |
|    time_elapsed    | 4        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 366         |
|    ep_rew_mean          | 0           |
| time/                   |             |
|    fps                  | 352         |
|    iterations           | 2           |
|    time_elapsed         | 11          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.013522992 |
|    clip_fraction        | 0.145       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.7        |
|    explained_variance   | -0.4        |
|    learning_rate        | 0.

KeyboardInterrupt: 

# Impala

In [11]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model = PPO(ImpalaActorCriticPolicy, wrapped_env, verbose=1, device="cuda")
model.learn(5_000_000)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 390      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    fps             | 237      |
|    iterations      | 1        |
|    time_elapsed    | 8        |
|    total_timesteps | 2048     |
---------------------------------


KeyboardInterrupt: 

In [47]:
# cannot use the sym link to save
model.save("/storage/coda1/p-adelarue3/0/rmehta98/impala_sanity")

In [48]:
obs, _ = wrapped_env.reset()
model.predict(obs)

(array(8), None)

# Test Model

In [8]:
from utils import get_filename_with_highest_timestep

In [11]:
import os

path = "/storage/home/hcoda1/4/rmehta98/p-adelarue3-0/impala_10env_25m_easy"

checkpoint, _ = get_filename_with_highest_timestep(path)

In [12]:
from stable_baselines3 import PPO
from impala import ImpalaActorCriticPolicy

model_loaded = PPO.load(os.path.join(path, checkpoint), env=wrapped_env, verbose=True, policy=ImpalaActorCriticPolicy)

# model_loaded = PPO(ImpalaActorCriticPolicy, wrapped_env, verbose=1)
# model_loaded.load()

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.


In [13]:

for ep in range(20): # number of episodes
    terminated, truncated = False, False
    total_reward = 0
    obs, _ = wrapped_env.reset()
    step = 0
    while not terminated and not truncated:
        action, _ = model_loaded.predict(obs)
        # out = wrapped_env.step(action)
        obs, reward, truncated, terminated, _ = wrapped_env.step(action)
        total_reward += reward
        step += 1
    
    print(f"Episode {ep + 1} terminated with reward of {total_reward} (step {step})")

Episode 1 terminated with reward of 0.0 (step 989)
Episode 2 terminated with reward of 0.0 (step 1000)
Episode 3 terminated with reward of 0.0 (step 1000)
Episode 4 terminated with reward of 0.0 (step 1000)
Episode 5 terminated with reward of 0.0 (step 1000)
Episode 6 terminated with reward of 0.0 (step 1000)
Episode 7 terminated with reward of 0.0 (step 1000)
Episode 8 terminated with reward of 0.0 (step 1000)
Episode 9 terminated with reward of 0.0 (step 1000)
Episode 10 terminated with reward of 0.0 (step 1000)
Episode 11 terminated with reward of 0.0 (step 1000)
Episode 12 terminated with reward of 0.0 (step 1000)
Episode 13 terminated with reward of 0.0 (step 1000)
Episode 14 terminated with reward of 0.0 (step 1000)
Episode 15 terminated with reward of 0.0 (step 1000)
Episode 16 terminated with reward of 0.0 (step 1000)
Episode 17 terminated with reward of 0.0 (step 1000)
Episode 18 terminated with reward of 0.0 (step 1000)
Episode 19 terminated with reward of 0.0 (step 1000)
Epi